In [1]:
cd ..

/Users/camila.cusicanqui/Documents/GitHub/frod-agentic-ai


In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
%pip install -e /Users/camila.cusicanqui/Documents/GitHub/analytics-mage-infra/mage-fraud-space

Obtaining file:///Users/camila.cusicanqui/Documents/GitHub/analytics-mage-infra/mage-fraud-space

[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
ERROR: file:///Users/camila.cusicanqui/Documents/GitHub/analytics-mage-infra/mage-fraud-space does not appear to be a Python project: neither 'setup.py' nor 'pyproject.toml' found.
Note: you may need to restart the kernel to use updated packages.


In [ ]:
# homemade utils
from utils.data_ingest import get_db_conn
from utils.chargeback_data import load_cb_df
# librerías 
import pandas as pd
import numpy as np
import gspread
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import date, timedelta, datetime
import gspread
from dateutil.utils import today

# formatting options
pd.set_option('display.max_columns', None)
pd.options.display.float_format = '{:,.2f}'.format


In [6]:
cb_df = load_cb_df()

/Users/camila.cusicanqui/Documents/GitHub/frod-agentic-ai/utils/chargeback_data.py:201: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, conn)


In [11]:
cb_df["amount_pos"] = cb_df["amount"]*-1


In [15]:
important_cols = [
    'cb_timestamp',
    'user_id',
    'transaction_id',
    'trx_timestamp_mx',
    'amount_pos',
    # 'merchant',
    'operador',
    'mcc_code',
    'card_type',
    'product_type',
    'nombrearchivo',
    'pos_entry_mode',
    'cvv_ind',
    'metodo_identificacion',
    'three_ds_flow',
    'afiliacion',
    'adquirente',
    'country',
    'cod_respuesta',   
]

In [17]:
cb_df[important_cols]

,cb_timestamp,user_id,transaction_id,trx_timestamp_mx,amount_pos,operador,mcc_code,card_type,product_type,nombrearchivo,pos_entry_mode,cvv_ind,metodo_identificacion,three_ds_flow,afiliacion,adquirente,country,cod_respuesta
0,2025-06-03 22:44:13,44f3688e-45fd-44ea-81fe-b22d8c7cf1d8,PARABILIUM:100159166,2025-05-18 21:57:55.570,800.00,OPENPAY*ADMA PROVIDENCIA 1JALMX,7277,VIRTUAL,CREDIT_5401_BIN,NaN,CNP Manual,has cvv,NaN,CHALLENGE,4200578,12,MX,00
1,2025-05-21 15:12:43,18918cbe-d254-425a-8a11-9b7cf98d8b4d,PARABILIUM:100463184,2025-05-19 22:28:42.643,"2,774.00",VOLARIS MOTO DEB 2 CIUDAD DE MEX001MX,4511,PHYSICAL,PLATINUM,par250125ProdJSP02_SubEm0004.emb,CNP Manual,has cvv,NaN,UNKNOWN,8112910,5,MX,00
2,2025-06-09 01:47:29,e0b3bdb2-ec5f-4784-a5d4-9950225960c9,PARABILIUM:100679780,2025-05-20 18:30:51.097,109.00,TRACTEBEL DGQRO MU MEXICO DF DF MX,5983,PHYSICAL,CREDIT_5401_BIN,par240403ProdJSO01_SubEm0001.emb,CNP Manual,has cvv,NaN,FRICTIONLESS,4040922,12,MX,00
3,2025-05-26 20:50:10,f8fb6af7-bb15-40e2-a06a-3206ff35a40f,101015177,2025-05-21 20:01:21.900,349.00,KFC ECOMMERCE CIUDAD DE MEX001MX,5814,VIRTUAL,CREDIT_5401_BIN,NaN,CNP Manual,has cvv,NaN,CHALLENGE,8419607,302,MX,00
4,2025-05-28 14:26:40,69cc717e-f27b-4a0f-b46f-3315d16fd228,PARABILIUM:101804609,2025-05-24 13:16:47.739,263.00,SERV CONSULTORIA MK HUIXQUILUCAN 015MX,7392,VIRTUAL,CREDIT_5456_BIN,NaN,CNP Manual,no cvv,NaN,UNKNOWN,9628274,485,MX,00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
273751,2025-10-24 22:28:43,b69e2053-e9de-4cf7-af2f-350209fa7e8b,c09c3cd9-1670-4fbd-8010-29bfe1917543:388852:SI...,2025-10-21 18:33:24.832,149.00,NaN,0731,PHYSICAL,PLATINUM,par250812ProdJSP01_SubEm0004.emb,NaN,unknown,NaN,NaN,NaN,NaN,NaN,NaN
273752,2025-09-24 23:10:48,3d2a5350-fa17-4a11-9d00-1e08e2431976,dd56284b-bf95-4912-ab11-083c659fb081:970959:88...,2025-09-24 16:02:31.436,367.53,NaN,0412,PHYSICAL,CREDIT_5401_BIN,par250916ProdJSO02_SubEm0001.emb,NaN,unknown,NaN,NaN,NaN,NaN,NaN,NaN
273753,2025-03-21 20:16:49,14a6bb69-4303-4e85-900a-67a1c8425ce3,ed4a9f04-6c8b-4a7c-9aac-4cda56b42e15: :00...,2025-03-05 17:59:41.078,721.50,NaN,0507,PHYSICAL,CREDIT_5401_BIN,par241124ProdJSO01_SubEm0001.emb,NaN,unknown,NaN,NaN,NaN,NaN,NaN,NaN
273754,2025-09-09 12:35:59,cba1d2c0-6bdd-42ad-b753-e6ae94fcffae,f60976f9-0391-404d-a390-42ea83e3ecc5:830569:3O...,2025-09-08 18:01:16.906,99.00,NaN,0489,PHYSICAL,CREDIT_5456_BIN,NaN,NaN,unknown,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
cb_df.groupby(['three_ds_status','three_ds_flow']).transaction_id.nunique()

three_ds_status        three_ds_flow 
3DS_AUTHENTICATED      CHALLENGE          10993
                       FRICTIONLESS        2987
3DS_NOT_AUTHENTICATED  EXEMPT_OR_INFO        52
UNKNOWN                UNKNOWN           241569
Name: transaction_id, dtype: int64

In [9]:
import asyncio
#from agents import Agent, Runner
#from agents import Agent, Runner, function_tool
import datetime as dt
import polars as pl
# import docx2txt
from pydantic import BaseModel, Field
from typing import List, Optional

In [18]:
from fraud_agents.schemas.transactions import TransactionRow

ModuleNotFoundError: No module named 'fraud_agents'

In [ ]:
import json
records = cb_df[important_cols].where(cb_df[important_cols].notna(), None).to_dict("records")

# 2) validate rows with pydantic
batch = TransactionBatch(
    transactions=[TransactionRow.model_validate(r) for r in records]
)

# 3) pass to prompt as JSON
user_prompt = f"""
Analyze these transactions for fraud patterns and summarize key findings.

Transactions:
{json.dumps(batch.model_dump(), ensure_ascii=False, default=str, indent=2)}
"""

In [10]:
import os
import yaml

with open(".config/credentials-mage.yaml", "r") as f:
    creds = yaml.safe_load(f)

for item in creds:
    os.environ[item["name"]] = str(item["value"])